## Import packages

In [ ]:
%pip install sdv pandas ipykernel

In [ ]:
import sdv
import pandas as pd

## Settings and files to process

In [ ]:
from pathlib import Path

DATA_FOLDER = Path('data')
OUTPUT_FOLDER = Path('data/synthetic')  # synthetic data and metadata are saved here
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)  # creates the folder if it doesn't exist

# Large files are randomly sampled down to this many rows for training,
# so memory use and run time stay manageable
MAX_TRAINING_ROWS = 50_000

# Rows of synthetic data to create for each file
NUM_SYNTHETIC_ROWS = 100

# Every CSV in the data folder, smallest first.
# To process only some files, list them instead, e.g.:
# csv_files = [DATA_FOLDER / 'customers.csv', DATA_FOLDER / 'orders.csv']
csv_files = sorted(DATA_FOLDER.glob('*.csv'), key=lambda f: f.stat().st_size)

In [ ]:
pd.DataFrame({
    'file': [f.name for f in csv_files],
    'size_MB': [round(f.stat().st_size / 1e6, 1) for f in csv_files],
})

## Helpers: load a training sample and detect metadata

In [ ]:
def count_rows(path):
    """Count data rows in a CSV quickly, without loading it into memory."""
    with open(path, 'rb') as f:
        return sum(block.count(b'\n') for block in iter(lambda: f.read(1 << 20), b'')) - 1


def load_training_sample(path, max_rows=MAX_TRAINING_ROWS, seed=0):
    """Read a CSV. Files with more than max_rows rows are read in chunks and
    sampled evenly, so the whole file is never held in memory at once."""
    total_rows = count_rows(path)
    if total_rows <= max_rows:
        return pd.read_csv(path, low_memory=False), total_rows
    fraction = max_rows / total_rows
    chunks = (chunk.sample(frac=fraction, random_state=seed)
              for chunk in pd.read_csv(path, chunksize=200_000))
    return pd.concat(chunks, ignore_index=True), total_rows

In [ ]:
from sdv.metadata import Metadata

# Fix column types that auto-detection gets wrong, per table (table name = file name without .csv), e.g.:
# COLUMN_FIXES = {
#     'customers': {'billing_address': 'address'},
# }
COLUMN_FIXES = {}


def build_metadata(table_name, df):
    """Auto-detect column types for one table, then apply any fixes above."""
    metadata = Metadata.detect_from_dataframes({table_name: df})
    for column, sdtype in COLUMN_FIXES.get(table_name, {}).items():
        metadata.update_column(column_name=column, sdtype=sdtype, table_name=table_name)
    metadata.validate()
    return metadata

## Create synthetic data for every file
Each file is loaded, modelled and saved one at a time. A file that fails is reported in the summary and the rest carry on.

In [ ]:
import time
from sdv.single_table import GaussianCopulaSynthesizer

summary = []

for path in csv_files:
    table_name = path.stem
    start = time.time()
    total_rows = training_rows = columns = None
    try:
        df, total_rows = load_training_sample(path)
        training_rows, columns = df.shape

        metadata = build_metadata(table_name, df)
        metadata.save_to_json(OUTPUT_FOLDER / f'{table_name}_metadata.json', mode='overwrite')

        synthesizer = GaussianCopulaSynthesizer(metadata)
        synthesizer.fit({table_name: df})
        synthetic = synthesizer.sample(num_rows=NUM_SYNTHETIC_ROWS)[table_name]

        synthetic.to_csv(OUTPUT_FOLDER / f'{table_name}_synthetic.csv', index=False)
        status = 'ok'
    except Exception as e:
        status = f'failed: {type(e).__name__}: {e}'

    seconds = round(time.time() - start, 1)
    print(f'{table_name}: {status} ({seconds}s)')
    summary.append({'table': table_name, 'real_rows': total_rows, 'training_rows': training_rows,
                    'columns': columns, 'status': status, 'seconds': seconds})

In [ ]:
summary_df = pd.DataFrame(summary)
summary_df

In [ ]:
# Preview the synthetic data for one table
pd.read_csv(OUTPUT_FOLDER / f'{csv_files[0].stem}_synthetic.csv').head(10)

## ⚠️ Final checks:
Synthetic data doesn't mean data is 100% anonymous. There might still be some sensitive information that can be derived from the metadata or the synthetic records. 

Therefore, it's recommended to make sure this dataset is safe to share online. Pro tip: Use minimum data needed for your analysis - if a column is redundant, just remove it!